<a href="https://colab.research.google.com/github/martirossi/AppliedML2026_mr/blob/main/regression_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# MLP Regressor + SHAP Feature Selection + Final Test Predictions
# ============================================

import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ------------------------------------------------
# 1. Load training data (electrons only for regression)
# ------------------------------------------------
train_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_train.csv"
data = pd.read_csv(train_path)

# Keep only electrons for regression
data = data[data["p_Truth_isElectron"] == 1]

# Target: true electron energy
target_col = "p_Truth_Energy"
y = data[target_col].values

# Detect ID column if present
id_col = None
for col in data.columns:
    if col.lower() in ["eventid", "id"]:
        id_col = col

# Feature columns = all except target and ID
feature_cols = [c for c in data.columns if c not in [target_col, id_col]]

# Full feature matrix (electrons only)
X_full = data[feature_cols].values






In [3]:
# ------------------------------------------------
# 2. Temporary XGBoost model for SHAP feature ranking
# ------------------------------------------------
temp_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist"
)

temp_model.fit(X_full, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [4]:
# ------------------------------------------------
# 3. SHAP feature importance
# ------------------------------------------------
explainer = shap.TreeExplainer(temp_model)
shap_values = explainer.shap_values(X_full)

shap_importance = np.abs(shap_values).mean(axis=0)

shap_ranking = pd.DataFrame({
    "Feature": feature_cols,
    "SHAP_Importance": shap_importance
}).sort_values(by="SHAP_Importance", ascending=False)

top_20_features = shap_ranking["Feature"].head(20).tolist()

print("Selected 20 features using SHAP:")
for f in top_20_features:
    print(f)

Selected 20 features using SHAP:
pX_maxEcell_energy
pX_ecore
p_pt_track
p_etcone20
p_ptcone40
pX_e233
pX_E3x5_Lr1
pX_E_Lr2_MedG
pX_E_Lr1_MedG
pX_MultiLepton
p_sigmad0
pX_E_Lr0_HiG
pX_E3x5_Lr0
pX_f1core
pX_deltaPhiFromLastMeasurement
p_ptPU30
pX_etcone20
pX_topoetcone20
pX_deltaPhi2
pX_E7x11_Lr1


In [5]:
# ------------------------------------------------
# 4. Prepare data with selected features
# ------------------------------------------------
X = data[top_20_features].values

In [7]:
# ------------------------------------------------
# 5. 5-fold CV with MLPRegressor + early stopping
#    + print MSE, RMSE, MAE, RelMAD per ogni fold
# ------------------------------------------------
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(data))

fold = 1
for train_idx, valid_idx in kf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=20,
        validation_fraction=0.1,
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predict on validation
    preds = model.predict(X_valid)
    oof_preds[valid_idx] = preds

    # 1) MSE
    mse = mean_squared_error(y_valid, preds)

    # 2) RMSE
    rmse = np.sqrt(mse)

    # 3) MAE
    mae = mean_absolute_error(y_valid, preds)

    # 4) RelMAD = mean( |(E_pred - E_true) / E_true| )
    rel_errors = np.abs((preds - y_valid) / y_valid)
    relmad = np.mean(rel_errors)

    print(f"Fold MSE   : {mse:.4f}")
    print(f"Fold RMSE  : {rmse:.4f} GeV")
    print(f"Fold MAE   : {mae:.4f} GeV")
    print(f"Fold RelMAD: {relmad:.4f}")

    fold += 1





===== Fold 1 =====
Fold MSE   : 338591267.3889
Fold RMSE  : 18400.8496 GeV
Fold MAE   : 8059.6900 GeV
Fold RelMAD: 0.3485

===== Fold 2 =====
Fold MSE   : 361872248.0799
Fold RMSE  : 19022.9400 GeV
Fold MAE   : 8501.5498 GeV
Fold RelMAD: 0.3559

===== Fold 3 =====
Fold MSE   : 679946435.0206
Fold RMSE  : 26075.7825 GeV
Fold MAE   : 8081.6719 GeV
Fold RelMAD: 0.3217

===== Fold 4 =====
Fold MSE   : 299581327.8545
Fold RMSE  : 17308.4178 GeV
Fold MAE   : 8188.4381 GeV
Fold RelMAD: 0.3164

===== Fold 5 =====
Fold MSE   : 348394182.3522
Fold RMSE  : 18665.3203 GeV
Fold MAE   : 8525.0013 GeV
Fold RelMAD: 0.3044


In [8]:
# ------------------------------------------------
# 6. Train final model on ALL data
# ------------------------------------------------
final_model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1,
    random_state=42
)

final_model.fit(X, y)

MLPRegressor(early_stopping=True, hidden_layer_sizes=(128, 64), max_iter=500,
             n_iter_no_change=20, random_state=42)

In [9]:
# ------------------------------------------------
# 7. Load test data and predict
# ------------------------------------------------
test_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_test_regression.csv"
test_data = pd.read_csv(test_path)

# Detect ID column
test_id_col = None
for col in test_data.columns:
    if col.lower() in ["eventid", "id"]:
        test_id_col = col
        break

if test_id_col is not None:
    test_ids = test_data[test_id_col].values
else:
    test_ids = np.arange(len(test_data))

X_test = test_data[top_20_features].values
test_preds = final_model.predict(X_test)

In [10]:
# ------------------------------------------------
# 8. Save required files
# ------------------------------------------------

# File 1: predictions
submission_df = pd.DataFrame({
    "Index": np.arange(len(test_preds)),
    "Predicted_Energy": test_preds
})
submission_df.to_csv("Regression_MartinaRossi_MLP.csv", index=False)

# File 2: variable list
varlist_df = pd.DataFrame({"FeatureName": top_20_features})
varlist_df.to_csv("Regression_MartinaRossi_MLP_VariableList.csv", index=False)

print("\nSaved files:")
print(" - Regression_MartinaRossi_MLP.csv")
print(" - Regression_MartinaRossi_MLP_VariableList.csv")


Saved files:
 - Regression_MartinaRossi_MLP.csv
 - Regression_MartinaRossi_MLP_VariableList.csv
